# Continuum: generate training data for a memory retriever

Builds (question → answering turn) training pairs from openly licensed conversations, for
fine-tuning a reranker that finds the right memory in a chat history.

1. **Stream conversations** from Conversation Chronicles (CC BY 4.0), Synthetic-Persona-Chat
   (CC BY 4.0) and OpenAssistant oasst2 (Apache 2.0). ShareGPT and UltraChat are deliberately
   excluded because LongMemEval, our test set, was built from them.
2. **Generate questions** with Qwen2.5-7B-Instruct (Apache 2.0): it picks lines holding a
   concrete fact (the owner's own lines, or the Assistant's in OpenAssistant chats) and writes
   the question the owner would ask later to recall it.
3. **Filter and verify**: cheap rules, then the same model checks each line really answers its question.
4. **Mine hard negatives** with bge-small: lines from the owner's memory of the same conversation
   that look relevant but aren't the answer. The model double-checks the top ones and drops any
   that actually answer the question.
5. **Contamination check**: drop anything that also appears in LongMemEval.

**How to run on Kaggle**
1. Settings → Accelerator: **GPU T4 x2**. Settings → Internet: **On**.
2. Leave `PILOT = True` and Run All. Read the printed samples and the time estimate.
3. If the samples look right, set `PILOT = False` and the `SHARD_COUNT` the estimate suggests.
   For each shard, set `SHARD_INDEX` and use **Save Version → Save & Run All**, so the run
   continues without the browser open. Each shard's files appear in that version's Output tab.

In [ ]:
import importlib.util
import os
import subprocess
import sys

import torch

SMOKE_TEST = os.environ.get("MEMORY_DATA_SMOKE_TEST") == "1"  # tiny local check

# With a single GPU the 7B model is loaded in 4-bit, which needs bitsandbytes.
LOAD_IN_4BIT = not SMOKE_TEST and torch.cuda.device_count() < 2
if LOAD_IN_4BIT and importlib.util.find_spec("bitsandbytes") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)

In [ ]:
import csv
import gc
import gzip
import hashlib
import io
import json
import random
import re
import time
from pathlib import Path

import numpy as np
import requests
from huggingface_hub import hf_hub_download, hf_hub_url
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

PILOT = True  # True: a few hundred questions to inspect; False: the full dataset

# The full run is split into shards that each fit in one Kaggle session. Shard i processes
# every SHARD_COUNT-th conversation starting at i, so the shards never overlap.
SHARD_COUNT = 1
SHARD_INDEX = 0

# Conversations to read from each source (totals across all shards). Chronicles gives 5
# sessions, so 5 prompts, per conversation; the others give 1.
PILOT_CONVERSATIONS = {"chronicles": 30, "persona_chat": 60, "oasst2": 40}
FULL_CONVERSATIONS = {"chronicles": 2000, "persona_chat": 6000, "oasst2": 1000}
MAX_HOURS_PER_SHARD = 5  # leaves headroom under Kaggle's 12-hour limit

GENERATOR_MODEL = "Qwen/Qwen2.5-7B-Instruct"
RETRIEVER_MODEL = "BAAI/bge-small-en-v1.5"  # Continuum's retriever, used to mine hard negatives
RETRIEVER_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

QUESTIONS_PER_PROMPT = 3
MAX_LINE_CHARS = 700  # long turns are shortened in prompts; stored training text stays whole
GENERATION_BATCH_SIZE = 16
VERIFY_BATCH_SIZE = 32
NUM_HARD_NEGATIVES = 7
FALSE_NEGATIVE_MARGIN = 0.95  # drop "negatives" scoring above 95% of the answer: likely also answers
FALSE_NEGATIVE_CHECKS = 3  # the model double-checks this many top negatives per example
VALIDATION_PERCENT = 5
SEED = 0

CHRONICLES_REPO = "jihyoung/ConversationChronicles"
PERSONA_CHAT_REPO = "google/Synthetic-Persona-Chat"
OASST_REPO = "OpenAssistant/oasst2"
LONGMEMEVAL_REPO = "xiaowu0162/longmemeval-cleaned"
LONGMEMEVAL_FILE = "longmemeval_s_cleaned.json"

if SMOKE_TEST:
    PILOT_CONVERSATIONS = {"chronicles": 2, "persona_chat": 3, "oasst2": 3}
    GENERATOR_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
    LONGMEMEVAL_FILE = "longmemeval_oracle.json"
    GENERATION_BATCH_SIZE = 4

CONVERSATIONS = PILOT_CONVERSATIONS if PILOT else FULL_CONVERSATIONS
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
random.seed(SEED)

print(f"{'Pilot' if PILOT else 'Full'} run: {CONVERSATIONS}, shard {SHARD_INDEX + 1} of {SHARD_COUNT}")
print(f"GPUs: {torch.cuda.device_count()} | generator: {GENERATOR_MODEL}" + (" (4-bit)" if LOAD_IN_4BIT else ""))

## Stream conversations
Files are read as streams and stop after the needed number of conversations, so nothing
large is downloaded. Each conversation becomes a list of sessions of (speaker, text) turns,
plus the **owner**: the person whose memory the questions are about.

In [ ]:
def stream_text(repo, filename, compressed=False):
    """Yield lines from a dataset file on Hugging Face without downloading all of it."""
    response = requests.get(hf_hub_url(repo, filename, repo_type="dataset"), stream=True, timeout=120)
    response.raise_for_status()
    if compressed:
        lines = io.TextIOWrapper(gzip.GzipFile(fileobj=response.raw), encoding="utf-8")
    else:
        response.raw.decode_content = True
        lines = io.TextIOWrapper(response.raw, encoding="utf-8", newline="")
    try:
        yield from lines
    finally:
        response.close()


SESSION_NAMES = ["first", "second", "third", "fourth", "fifth"]


def load_chronicles(limit):
    """Five sessions between two people, hours to years apart."""
    for index, line in enumerate(stream_text(CHRONICLES_REPO, "train.jsonl")):
        if index >= limit:
            return
        row = json.loads(line)
        sessions = [
            [(speaker, text.strip()) for speaker, text in zip(row[f"{name}_session_speakers"], row[f"{name}_session_dialogue"]) if text.strip()]
            for name in SESSION_NAMES
        ]
        sessions = [session for session in sessions if session]
        speakers = list(dict.fromkeys(speaker for session in sessions for speaker, _ in session))
        if sessions:
            # Alternate whose memory it is, so both roles (Parent/Child, A/B, ...) appear.
            owner = speakers[index % len(speakers)]
            yield {"id": f"chronicles/{row['dataID']}", "source": "chronicles", "owner": owner, "sessions": sessions}


def load_persona_chat(limit):
    """Two people with given personas getting to know each other."""
    rows = csv.DictReader(stream_text(PERSONA_CHAT_REPO, "data/Synthetic-Persona-Chat_train.csv"))
    for index, row in enumerate(rows):
        if index >= limit:
            return
        turns = []
        for line in row["Best Generated Conversation"].splitlines():
            match = re.match(r"\s*(User [12])\s*:\s*(.*)", line)
            if match:
                turns.append([match.group(1), match.group(2).strip()])
            elif turns and line.strip():
                turns[-1][1] += " " + line.strip()  # continuation of the previous turn
        turns = [(speaker, text) for speaker, text in turns if text]
        if len(turns) >= 4:
            owner = "User 1" if index % 2 == 0 else "User 2"  # alternate whose memory it is
            yield {"id": f"persona_chat/{index}", "source": "persona_chat", "owner": owner, "sessions": [turns]}


def best_path(message):
    """Follow the highest-ranked reply at each step of an OpenAssistant conversation tree."""
    path = [message]
    while message.get("replies"):
        message = min(message["replies"], key=lambda reply: 99 if reply.get("rank") is None else reply["rank"])
        path.append(message)
    return path


def load_oasst(limit):
    """Human-written user–assistant conversations (English only)."""
    count = 0
    for line in stream_text(OASST_REPO, "2023-11-05_oasst2_ready.trees.jsonl.gz", compressed=True):
        tree = json.loads(line)
        if tree["prompt"]["lang"] != "en":
            continue
        turns = [("User" if m["role"] == "prompter" else "Assistant", m["text"].strip()) for m in best_path(tree["prompt"])]
        turns = [turn for turn in turns if turn[1]]
        if len(turns) < 2:
            continue
        yield {"id": f"oasst2/{tree['message_tree_id']}", "source": "oasst2", "owner": "User", "sessions": [turns]}
        count += 1
        if count >= limit:
            return


LOADERS = {"chronicles": load_chronicles, "persona_chat": load_persona_chat, "oasst2": load_oasst}

conversations = []
for source, limit in CONVERSATIONS.items():
    loaded = [
        conversation
        for position, conversation in enumerate(LOADERS[source](limit))
        if position % SHARD_COUNT == SHARD_INDEX
    ]
    conversations.extend(loaded)
    turns = sum(len(session) for c in loaded for session in c["sessions"])
    print(f"{source:13s} {len(loaded):6,} conversations, {turns:8,} turns")

## Generate questions
One prompt per session. The model returns JSON naming the line that holds each fact.

In [ ]:
SYSTEM_PROMPT = "You create training data for a personal memory assistant. Follow the output format exactly."

QUESTION_PROMPT = """Below is a conversation with numbered lines. The memory owner is "{owner}".

Pick up to {count} lines spoken by {answer_speaker} that contain {fact_kind}. Never pick a line spoken by anyone else.

For each picked line, write the question {owner} might ask their memory assistant weeks later to recall it.

Rules:
- {perspective_rule}
- The picked line alone must answer the question.
- Paraphrase: avoid copying distinctive words from the line when natural wording exists.
- Skip greetings, small talk and lines without a concrete fact. Returning [] is fine.

Conversation:
{conversation}

Reply with JSON only, no other text: [{{"line": <line number>, "question": "<question>"}}]"""

PERSONAL_FACT = (
    "a specific fact {owner} shared about their own life "
    "(plans, events, preferences, possessions, people, places, dates)"
)
ASSISTANT_FACT = "specific advice or information the Assistant gave {owner}"


def answer_speaker(conversation):
    """Who may speak an answer line: the owner, or the Assistant in user–assistant chats."""
    return "Assistant" if conversation["source"] == "oasst2" else conversation["owner"]

PERSPECTIVE_RULE = (
    'Write the question as {owner} talking about themselves, in first person: use "I", "my" and "me", '
    'never "you" (e.g. "When is my sister\'s wedding?", "What did I decide about the job offer?").'
)
PERSPECTIVE_RULE_ASSISTANT = (
    'Write the question as {owner} talking to their assistant, in first person: use "I", "my" and "me"; '
    '"you" may only mean the Assistant (e.g. "What did you suggest for my sore back?").'
)

VERIFY_PROMPT = """Text (said by {speaker}): "{text}"

Question (asked later by {owner}): "{question}"

Does the text, on its own, contain the answer to the question? Reply with only "yes" or "no"."""


def shorten(text, limit=MAX_LINE_CHARS):
    return text if len(text) <= limit else text[:limit].rsplit(" ", 1)[0] + " ..."


prompts, prompt_sources = [], []  # prompt_sources[i] = (conversation index, session index)
for conversation_index, conversation in enumerate(conversations):
    for session_index, session in enumerate(conversation["sessions"]):
        numbered = "\n".join(f"[{i}] {speaker}: {shorten(text)}" for i, (speaker, text) in enumerate(session, start=1))
        with_assistant = conversation["source"] == "oasst2"
        owner = conversation["owner"]
        prompts.append(QUESTION_PROMPT.format(
            owner=owner,
            count=QUESTIONS_PER_PROMPT,
            answer_speaker="the Assistant" if with_assistant else owner,
            fact_kind=(ASSISTANT_FACT if with_assistant else PERSONAL_FACT).format(owner=owner),
            perspective_rule=(PERSPECTIVE_RULE_ASSISTANT if with_assistant else PERSPECTIVE_RULE).format(owner=owner),
            conversation=numbered,
        ))
        prompt_sources.append((conversation_index, session_index))

tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

load_options = {"device_map": "auto"}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_options["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
else:
    load_options["torch_dtype"] = torch.float16
generator = AutoModelForCausalLM.from_pretrained(GENERATOR_MODEL, **load_options)
generator.eval()


def generate(user_prompts, max_new_tokens, batch_size):
    """Greedy-generate a reply to each prompt, batching prompts of similar length."""
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
        for prompt in user_prompts
    ]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    replies = [""] * len(texts)
    for start in tqdm(range(0, len(order), batch_size), desc=f"generating ({len(texts)} prompts)"):
        indices = order[start:start + batch_size]
        inputs = tokenizer([texts[i] for i in indices], return_tensors="pt", padding=True).to(generator.device)
        with torch.inference_mode():
            output = generator.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        decoded = tokenizer.batch_decode(output[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        for i, reply in zip(indices, decoded):
            replies[i] = reply.strip()
    return replies


def parse_questions(reply, line_count):
    """Extract [{"line": n, "question": "..."}] from a reply, ignoring anything malformed."""
    match = re.search(r"\[.*\]", reply, re.DOTALL)
    if not match:
        return None
    try:
        items = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    parsed = []
    for item in items if isinstance(items, list) else []:
        if not isinstance(item, dict):
            continue
        line, question = item.get("line"), str(item.get("question", "")).strip()
        if isinstance(line, int) and 1 <= line <= line_count and question.endswith("?") and len(question) > 10:
            parsed.append((line - 1, question))
    return parsed


started = time.perf_counter()
replies = generate(prompts, max_new_tokens=60 * QUESTIONS_PER_PROMPT + 40, batch_size=GENERATION_BATCH_SIZE)
generation_seconds = time.perf_counter() - started

candidates, unparsable = [], 0
for (conversation_index, session_index), reply in zip(prompt_sources, replies):
    session = conversations[conversation_index]["sessions"][session_index]
    parsed = parse_questions(reply, len(session))
    if parsed is None:
        unparsable += 1
        continue
    for turn_index, question in parsed:
        candidates.append({"conversation": conversation_index, "session": session_index, "turn": turn_index, "question": question})

print(f"{len(prompts):,} prompts -> {len(candidates):,} questions in {generation_seconds / 60:.1f} min "
      f"({unparsable} unparsable replies)")

## Filter, then verify each question is answered by its turn
Cheap rules run first:
- the answer line must be the owner's own (or the Assistant's, in OpenAssistant chats),
  because a question about "my" life answered by someone else's line teaches the wrong thing;
- a question that just repeats its line teaches nothing;
- in a chat between two people "you" can't be right, because the memory assistant wasn't there
  (in OpenAssistant chats "you" means the assistant, which is fine);
- the question has to be about the owner at all.

In [ ]:
def turn_of(candidate):
    return conversations[candidate["conversation"]]["sessions"][candidate["session"]][candidate["turn"]]


def normalize(text):
    return " ".join(text.lower().split())


FIRST_PERSON = re.compile(r"\b(i|my|me|mine|myself)\b", re.IGNORECASE)
SECOND_PERSON = re.compile(r"\byou(r|rs|rself)?\b", re.IGNORECASE)


def rule_problem(candidate):
    question = candidate["question"]
    conversation = conversations[candidate["conversation"]]
    with_assistant = conversation["source"] == "oasst2"
    if turn_of(candidate)[0] != answer_speaker(conversation):
        return "answer not spoken by the owner (or Assistant)"
    if normalize(question.rstrip("?")) in normalize(turn_of(candidate)[1]):
        return "copied from its line"
    if not with_assistant and SECOND_PERSON.search(question):
        return "asks 'you' in a two-person chat"
    if not FIRST_PERSON.search(question) and not (with_assistant and SECOND_PERSON.search(question)):
        return "not about the owner"
    return None


rejected = {}
passing = []
for candidate in candidates:
    problem = rule_problem(candidate)
    if problem:
        rejected[problem] = rejected.get(problem, 0) + 1
    else:
        passing.append(candidate)
print(f"Rule filters: kept {len(passing):,} of {len(candidates):,} questions; rejected {rejected}")
candidates_before_rules, candidates = candidates, passing

verify_prompts = [
    VERIFY_PROMPT.format(
        speaker=turn_of(c)[0], text=shorten(turn_of(c)[1], 1500),
        owner=conversations[c["conversation"]]["owner"], question=c["question"],
    )
    for c in candidates
]
started = time.perf_counter()
verdicts = generate(verify_prompts, max_new_tokens=3, batch_size=VERIFY_BATCH_SIZE)
verify_seconds = time.perf_counter() - started

seen_questions = set()
verified = []
for candidate, verdict in zip(candidates, verdicts):
    key = (candidate["conversation"], candidate["question"].lower())
    if verdict.lower().startswith("yes") and key not in seen_questions:
        seen_questions.add(key)
        verified.append(candidate)

print(f"Verified {len(verified):,} of {len(candidates):,} questions ({len(verified) / max(1, len(candidates)):.0%}) "
      f"in {verify_seconds / 60:.1f} min")

## Mine hard negatives, then double-check them
Negatives come only from lines that would be in the owner's memory: the owner's own lines
(both sides in OpenAssistant chats). Another person saying "I do yoga too" can't be a negative,
because the reranker can't see who spoke a line. bge-small ranks the candidates against the
question; the best-scoring wrong ones become hard negatives, skipping lines right next to the
answer and any scoring above 95% of it. The model then checks the top few and drops any that
actually answer the question: a "wrong" line that is really right teaches the opposite of the truth.

In [ ]:
retriever = SentenceTransformer(RETRIEVER_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

memory_turns = {}  # conversation index -> [(session, turn, speaker, text)] held in the owner's memory
for candidate in verified:
    index = candidate["conversation"]
    if index not in memory_turns:
        conversation = conversations[index]
        memory_turns[index] = [
            (s, t, speaker, text)
            for s, session in enumerate(conversation["sessions"])
            for t, (speaker, text) in enumerate(session)
            if conversation["source"] == "oasst2" or speaker == conversation["owner"]
        ]

turn_texts = [text for turns in memory_turns.values() for *_, text in turns]
turn_vectors = retriever.encode(turn_texts, batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
vectors_by_conversation, offset = {}, 0
for index, turns in memory_turns.items():
    vectors_by_conversation[index] = turn_vectors[offset:offset + len(turns)]
    offset += len(turns)

query_vectors = retriever.encode(
    [RETRIEVER_QUERY_PREFIX + c["question"] for c in verified],
    batch_size=128, normalize_embeddings=True, convert_to_numpy=True,
)

examples = []
for candidate, query_vector in zip(verified, query_vectors):
    turns = memory_turns[candidate["conversation"]]
    scores = vectors_by_conversation[candidate["conversation"]] @ query_vector
    position = next(i for i, (s, t, _, _) in enumerate(turns) if s == candidate["session"] and t == candidate["turn"])
    positive = turns[position][3]
    ceiling = scores[position] - (1 - FALSE_NEGATIVE_MARGIN) * abs(scores[position])

    negatives = []  # (speaker, text), hardest first
    for i in np.argsort(-scores):
        session, turn, speaker, text = turns[i]
        is_neighbour = session == candidate["session"] and abs(turn - candidate["turn"]) <= 1
        if i == position or is_neighbour or scores[i] >= ceiling or text == positive or any(text == t for _, t in negatives):
            continue
        negatives.append((speaker, text))
        if len(negatives) == NUM_HARD_NEGATIVES:
            break

    conversation = conversations[candidate["conversation"]]
    examples.append({
        "query": candidate["question"],
        "positive": positive,
        "negatives": negatives,
        "source": conversation["source"],
        "conversation_id": conversation["id"],
        "speaker": turn_of(candidate)[0],
        "owner": conversation["owner"],
    })

del retriever
gc.collect()
torch.cuda.empty_cache()

checks = [  # (example index, negative index) for each example's hardest negatives
    (e, n)
    for e, example in enumerate(examples)
    for n in range(min(FALSE_NEGATIVE_CHECKS, len(example["negatives"])))
]
check_prompts = [
    VERIFY_PROMPT.format(
        speaker=examples[e]["negatives"][n][0], text=shorten(examples[e]["negatives"][n][1], 1500),
        owner=examples[e]["owner"], question=examples[e]["query"],
    )
    for e, n in checks
]
started = time.perf_counter()
check_verdicts = generate(check_prompts, max_new_tokens=3, batch_size=VERIFY_BATCH_SIZE)
negative_check_seconds = time.perf_counter() - started

false_negatives = {check for check, verdict in zip(checks, check_verdicts) if verdict.lower().startswith("yes")}
for e, example in enumerate(examples):
    example["negatives"] = [text for n, (_, text) in enumerate(example["negatives"]) if (e, n) not in false_negatives]

del generator
gc.collect()
torch.cuda.empty_cache()

average_negatives = np.mean([len(e["negatives"]) for e in examples]) if examples else 0.0
print(f"{len(examples):,} examples. Checked {len(checks):,} top negatives and dropped {len(false_negatives):,} "
      f"that actually answer their question ({negative_check_seconds / 60:.1f} min); "
      f"{average_negatives:.1f} hard negatives left per example on average")

## Contamination check against LongMemEval
Any turn that also appears in LongMemEval is removed, so the model is never trained on test text.

In [ ]:
longmemeval =json.loads(Path(hf_hub_download(LONGMEMEVAL_REPO, LONGMEMEVAL_FILE, repo_type="dataset")).read_text(encoding="utf-8"))
test_turns = {normalize(turn["content"]) for item in longmemeval for session in item["haystack_sessions"] for turn in session}
test_questions = {normalize(item["question"]) for item in longmemeval}
del longmemeval

clean_examples, dropped, negatives_removed = [], 0, 0
for example in examples:
    if normalize(example["positive"]) in test_turns or normalize(example["query"]) in test_questions:
        dropped += 1
        continue
    kept = [n for n in example["negatives"] if normalize(n) not in test_turns]
    negatives_removed += len(example["negatives"]) - len(kept)
    clean_examples.append({**example, "negatives": kept})

print(f"LongMemEval overlap: dropped {dropped} examples and {negatives_removed} negatives "
      f"(checked against {len(test_turns):,} test turns)")

## Save and inspect

In [ ]:
def is_validation(conversation_id):
    return int(hashlib.md5(conversation_id.encode()).hexdigest(), 16) % 100 < VALIDATION_PERCENT


splits = {"train": [], "validation": []}
for example in clean_examples:
    splits["validation" if is_validation(example["conversation_id"]) else "train"].append(example)

for name, rows in splits.items():
    with open(OUTPUT_DIR / f"memory_retrieval_{name}_shard{SHARD_INDEX}.jsonl", "w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")

def count_by_source(items, source_of):
    return {source: sum(source_of(item) == source for item in items) for source in CONVERSATIONS}


candidate_source = lambda c: conversations[c["conversation"]]["source"]
yield_by_source = {
    "prompts": count_by_source(prompt_sources, lambda p: conversations[p[0]]["source"]),
    "questions": count_by_source(candidates_before_rules, candidate_source),
    "after rules": count_by_source(candidates, candidate_source),
    "verified": count_by_source(verified, candidate_source),
    "examples": count_by_source(clean_examples, lambda e: e["source"]),
}

print("Yield by source (use this to rebalance FULL_CONVERSATIONS):")
print(f"{'source':14s}" + "".join(f"{stage:>13s}" for stage in yield_by_source) + f"{'per prompt':>13s}")
for source in CONVERSATIONS:
    counts = [yield_by_source[stage][source] for stage in yield_by_source]
    per_prompt = counts[-1] / max(1, counts[0])
    print(f"{source:14s}" + "".join(f"{count:>13,}" for count in counts) + f"{per_prompt:>13.2f}")
print()

model_seconds = generation_seconds + verify_seconds + negative_check_seconds

stats = {
    "pilot": PILOT,
    "shard": {"index": SHARD_INDEX, "count": SHARD_COUNT},
    "generator": GENERATOR_MODEL,
    "conversations": {source: sum(c["source"] == source for c in conversations) for source in CONVERSATIONS},
    "prompts": len(prompts),
    "questions_generated": len(candidates_before_rules),
    "rule_rejections": rejected,
    "questions_verified": len(verified),
    "negatives_checked": len(checks),
    "false_negatives_dropped": len(false_negatives),
    "average_negatives": round(float(average_negatives), 2),
    "yield_by_source": yield_by_source,
    "contamination_dropped": dropped,
    "examples": {name: len(rows) for name, rows in splits.items()},
    "generation_minutes": round(generation_seconds / 60, 1),
    "verification_minutes": round(verify_seconds / 60, 1),
    "negative_check_minutes": round(negative_check_seconds / 60, 1),
}
(OUTPUT_DIR / f"memory_retrieval_stats_shard{SHARD_INDEX}.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
print(json.dumps(stats, indent=2))

if PILOT:
    # Scale this run's time and per-source yield up to FULL_CONVERSATIONS. Prompt lengths vary
    # by source, so treat this as a rough guide.
    seconds_per_prompt = model_seconds / max(1, len(prompts))
    full_prompts = {
        source: limit * (len(SESSION_NAMES) if source == "chronicles" else 1) for source, limit in FULL_CONVERSATIONS.items()
    }
    full_examples = sum(
        yield_by_source["examples"][source] / max(1, yield_by_source["prompts"][source]) * full_prompts[source]
        for source in full_prompts
    )
    full_hours = seconds_per_prompt * sum(full_prompts.values()) / 3600
    shards = max(1, int(np.ceil(full_hours / MAX_HOURS_PER_SHARD)))
    print(f"\nFull run estimate: ~{sum(full_prompts.values()):,} prompts, ~{full_hours:.1f} hours of model time, "
          f"~{full_examples:,.0f} examples.")
    print(f"Run it as SHARD_COUNT = {shards} shards of ~{full_hours / shards:.1f} hours each "
          f"(plus ~15 min of downloads per shard), with SHARD_INDEX = 0 to {shards - 1}.")

print("\nRandom samples to read before the full run:\n")
for example in random.sample(clean_examples, min(15, len(clean_examples))):
    print(f"[{example['source']}, owner: {example['owner']}] Q: {example['query']}")
    print(f"   answer ({example['speaker']}): {shorten(example['positive'], 220)}")
    if example["negatives"]:
        print(f"   hardest wrong line: {shorten(example['negatives'][0], 160)}")
    print()

## What to check in the samples
- **The answer is spoken by the owner** (or the Assistant in OpenAssistant chats), and it really
  answers the question.
- **Questions sound like the owner asking later**, in first person, without just copying the line.
  Some overlap is natural; constant copying would teach word matching instead of meaning.
- **The hardest wrong line is plausible but wrong.** If many still answer the question, raise
  `FALSE_NEGATIVE_CHECKS` or lower `FALSE_NEGATIVE_MARGIN`.

If the pilot looks good, set `PILOT = False` and the suggested `SHARD_COUNT`, then save and run
one version per `SHARD_INDEX`. The training notebook reads every
`memory_retrieval_{train,validation}_shard*.jsonl` file.